In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from config import load_config
from datasets.sudoscan_loader import SudoscanDataset
from models.model_factory import get_model
from backends.backend_adapter import get_dice_components
from dice_interface.dice_wrapper import DiCEWrapper
from module.explainers.shap_explainer import explain_with_shap

%matplotlib inline

In [ ]:
# Load config
config = load_config("config.yaml")

loader = SudoscanDataset()
dataset = loader.get_dataset()

# Load & preprocess only the selected features
x_train, x_test, y_train, y_test = loader.split(
    dataset=dataset,
    train_percentage=80
)

In [ ]:
# Build model with selected features
model = get_model(
    preprocessor=ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=
            [
                ('imputer', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler())
            ]
            ),
             loader.numeric_cols
             ),
            ('cat', Pipeline(steps=
            [
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]
            ),
             loader.categorical_cols
             )
        ]
    ),
    model_type=config["model_1"]["type"],
    backend=config["model_1"]["backend"],
    x_train=x_train,
    y_train=y_train
)

In [ ]:
dice_model, dice_data = get_dice_components(
    model=model,
    backend=config["model_1"]["backend"],
    x_train=x_train,
    y_train=y_train,
    input_features=config["dataset"]["feature_set_1"],
    continuous_features=loader.numeric_cols.tolist(),
    target=config["dataset"]["target"]
)

In [ ]:
# Generate counterfactuals
dice = DiCEWrapper(dice_model, dice_data)

# After you have trained your model...
y_preds = model.predict(x_test)

# Generate batches of CFs for the whole test set
batched_cfs = dice.generate_batched(
    query_instances=x_test,
    predictions=y_preds,
    features_to_vary=dataset.drop(['SEX', 'NS', 'CAS (%)', 'DPN_Status', ],
                                       axis=1).columns.tolist(),
    total_CFs=20
)

In [ ]:
for idx, cfs in batched_cfs.items():
    print(f"Counterfactual examples for target class {idx}")
    cfs.visualize_as_dataframe(show_only_changes=True)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_true = y_test.tolist()
y_pred = model.predict(x_test).tolist()

labels = [0, 1, 2, 3]

# Overall confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels)

# Per-class metrics
recall_list = []
specificity_list = []
youden_list = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FN = sum(cm[i, :]) - TP
    FP = sum(cm[:, i]) - TP
    TN = cm.sum() - (TP + FN + FP)

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    youden = recall + specificity - 1

    recall_list.append(recall)
    specificity_list.append(specificity)
    youden_list.append(youden)

# Macro average
print("Macro Recall:", np.mean(recall_list))
print("Macro Specificity:", np.mean(specificity_list))
print("Macro Youden Index:", np.mean(youden_list))

In [ ]:
for i, label in enumerate(labels):
    TP = cm[i, i]
    FN = sum(cm[i, :]) - TP
    FP = sum(cm[:, i]) - TP
    TN = cm.sum() - (TP + FN + FP)

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    youden = recall + specificity - 1

    recall_list.append(recall)
    specificity_list.append(specificity)
    youden_list.append(youden)

In [ ]:
print("\nPer-Class Metrics:")
for idx, label in enumerate(labels):
    print(f"Class {label}: "
          f"Recall={recall_list[idx]:.3f} | "
          f"Specificity={specificity_list[idx]:.3f} | "
          f"Youden Index={youden_list[idx]:.3f}")

In [ ]:
import pandas as pd

print(len(labels))
print(len(recall_list))
print(len(specificity_list))
print(len(youden_list))


df_metrics = pd.DataFrame({
    # 'Class': labels,
    'Recall': recall_list,
    'Specificity': specificity_list,
    'Youden Index': youden_list
})
print(df_metrics)

In [ ]:
print("Macro Recall:", np.mean(recall_list))
print("Macro Specificity:", np.mean(specificity_list))
print("Macro Youden Index:", np.mean(youden_list))

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# 1) True labels
y_true = y_test.values  # or y_test.to_numpy()

# 2) Probabilities from your ordinal model
y_probs = model.predict_proba(x_test)

# 3) Binarize labels: shape [n_samples, n_classes]
classes = np.unique(y_true)
y_true_bin = label_binarize(y_true, classes=classes)

# 4) Plot ROC curve for each class
plt.figure(figsize=(8, 6))

for i, class_label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'Class {class_label} AUC = {roc_auc:.2f}')

# 5) Chance line
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Multiclass ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# --- Your true labels & predicted probabilities ---
y_true = y_test.values  # or y_test.to_numpy()
y_probs = model.predict_proba(x_test)

# --- Binarize the multiclass labels ---
classes = np.unique(y_true)
y_true_bin = label_binarize(y_true, classes=classes)

# --- For each class, plot its own ROC ---
for i, class_label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Recall)')
    plt.title(f'ROC Curve for Class {class_label}')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

In [ ]:
# CELL 1: Descriptive summaries for each CF example
def summarize_cf_example(cfs):
    cf_summaries = []
    
    for idx, cf in enumerate(cfs.cf_examples_list):
        cf_df = cf.final_cfs_df
    
        # Filter numeric & categorical columns by features_to_vary
        numeric_features = [col for col in loader.numeric_cols if col in loader.get_features_to_vary()]
        categorical_features = [
            col for col in cf_df.select_dtypes(include=['object', 'category']).columns
            if col in loader.get_features_to_vary()
        ]
    
        # Numeric summary
        numeric_summary = cf_df[numeric_features].describe().T[
            ['mean', '50%', 'std', 'min', 'max']
        ].rename(columns={'50%': 'median'})
    
        # Categorical summary
        categorical_summary = {
            col: cf_df[col].mode().iloc[0] for col in categorical_features
        }
    
        cf_summaries.append({
            'index': idx,
            'cf_df': cf_df,
            'numeric_summary': numeric_summary,
            'categorical_summary': categorical_summary
        })
    
        print(f"CF Example {idx} — Numeric Summary (varied features only)")
        display(numeric_summary)

        print(f"CF Example {idx} — Categorical Summary (varied features only)")
        display(categorical_summary)

In [ ]:
for idx, cf in batched_cfs.items():
    print(f"Descriptive Summary of class {idx - 1} to {idx}")
    summarize_cf_example(cf)

In [ ]:
from copy import deepcopy

def merge_dice_results(cf_list):
    # Deepcopy the first one to serve as base
    merged_cf = deepcopy(cf_list.get(1))

    # Combine the rest
    for idx, cf in cf_list.items():
        merged_cf.cf_examples_list.extend(cf.cf_examples_list)

    return merged_cf

merged_cfs = merge_dice_results(batched_cfs)

In [ ]:
imp = dice.dice.global_feature_importance(x_test, cf_examples_list=merged_cfs.cf_examples_list)
print(imp.summary_importance, '\n')

In [ ]:
import matplotlib.pyplot as plt

# Sort features by importance (optional but useful for clarity)
sorted_items = sorted(imp.summary_importance.items(), key=lambda x: x[1], reverse=True)
features, importances = zip(*sorted_items)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(features, importances, color='skyblue')
plt.xlabel("Global Feature Importance")
plt.title("DiCE Global Counterfactual Feature Importance")
plt.gca().invert_yaxis()  # Highest on top
plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict

# Step 1: Group x_test by predicted class
from collections import defaultdict
import pandas as pd

# Make sure x_test is a DataFrame and y_preds is a Series or array
x_test = pd.DataFrame(x_test)
y_preds = pd.Series(y_preds)

# Group x_test by prediction
grouped_instances = defaultdict(list)
for idx, pred in enumerate(y_preds):
    grouped_instances[pred].append(idx)

# Step 2: Prepare to collect importance scores
from dice_ml import Dice

sum_importances = defaultdict(float)
count = 0

# Step 3: Loop through each predicted class group
for pred_class, indices in grouped_instances.items():
    target_class = pred_class + 1  # Where the CFs should belong

    if target_class not in batched_cfs:
        continue  # Skip if no CFs were generated for this transition

    cf = batched_cfs[target_class]
    cf_examples = cf.cf_examples_list
    x_batch = x_test.iloc[indices]

    for i, idx in enumerate(indices):
        query_instance = x_test.iloc[[idx]]  # Keep as DataFrame
        cf_example = cf_examples[i]

        # Get local importance
        imp = dice.dice.local_feature_importance(query_instances=query_instance,
                                                 cf_examples_list=[cf_example])
        local_imp = imp.local_importance

        # Accumulate
        for feature, score in local_imp.items():
            sum_importances[feature] += score

        count += 1

# Step 4: Average the scores
avg_importances = {feature: total / count for feature, total in sum_importances.items()}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Assuming `y_test` and `y_pred` are your true and predicted

In [ ]:
import shap

# Define a simple predict function:
def mord_predict(X):
    return model.predict(X)

# Use a masker: your input DataFrame
masker = shap.maskers.Independent(x_test)

# Create explainer with custom predict function
explainer = shap.Explainer(mord_predict, masker=masker)

# Compute SHAP values
shap_values = explainer(x_test)

# Plot
shap.summary_plot(shap_values, x_test)
